# Prepare Granite for GPU Serving with LLM Compressor

Your team plans to serve the IBM Granite 4.0 350M model on a Red Hat OpenShift AI cluster with GPUs.
Before you request GPU resources, prepare two quantized variants of the model, confirm that both variants still produce useful answers, and estimate how much GPU memory (VRAM) each variant needs.

You create the following variants:

* **W8A16**: 8-bit integer weights, 16-bit activations.
* **W4A16**: 4-bit integer weights, 16-bit activations.

Both schemes are *data-free*: LLM Compressor quantizes the weights directly, without a calibration dataset.
This workbench has no GPU, so you run the quantization and the quality checks on CPU.
The quantized models are ready to be deployed to GPU-backed serving runtimes later.

### 1. Import libraries

In [ ]:
import json
from pathlib import Path

from transformers import AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

print("✓ Libraries imported")

### 2. Configure paths and test prompts

The original model is mounted read-only from shared storage.
The quantized models are saved in the `quantized-models` directory, next to this notebook.
The test prompts check general knowledge, reasoning, and code generation.

In [ ]:
MODEL_PATH = Path("/shared-models/granite-4.0-350m")
OUTPUT_DIR = Path("quantized-models")

PROMPTS = [
    "The capital of France is",
    "Explain in one sentence why the sky is blue.",
    "Write a Python function that returns the square of a number.",
]

results = {}
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("✓ Configuration ready")

### 3. Import helper functions

The `compression_utils.py` file, next to this notebook, defines the following helper functions:

* `load_model()` loads a model from disk.
* `generate()` sends each test prompt to a model and returns the answers.
* `weights_size_mb()` returns the size of the model weights files (`*.safetensors`).
* `print_answers()` prints each test prompt with its answer.
* `reload_and_test()` reloads a saved model from disk, verifies that it answers the test prompts, and returns the results.

In [ ]:
from compression_utils import generate, load_model, print_answers, reload_and_test, weights_size_mb

print("✓ Helper functions imported")

### 4. Test the original model

Record the size and the answers of the original BF16 model as the baseline for the comparison.

In [ ]:
original = load_model(MODEL_PATH)
results["BF16"] = {
    "path": MODEL_PATH,
    "size_mb": weights_size_mb(MODEL_PATH),
    "answers": generate(original, tokenizer, PROMPTS),
}
del original

print(f"✓ Original model: {results['BF16']['size_mb']:.0f} MB\n")
print_answers(PROMPTS, results["BF16"]["answers"])

### 5. Recipe 1: W8A16

This model shares the weights of its output layer, `lm_head`, with the input embeddings, so keep `lm_head` at the original precision by adding it to the `ignore` list.

Complete the TODO to create a recipe that quantizes all `Linear` layers with the W8A16 scheme.

After the recipe cell, the following cell loads a fresh copy of the original model, applies the recipe with `oneshot()`, saves the compressed model, and then reloads it from disk to verify that it works.

In [ ]:
recipe_w8a16 = None

# TODO: replace the previous line with a recipe that quantizes all Linear layers
# with the W8A16 scheme and ignores the lm_head layer:
#
# recipe_w8a16 = QuantizationModifier(
#     targets="Linear",
#     scheme="...",
#     ignore=["lm_head"],
# )

In [ ]:
assert recipe_w8a16 is not None, "Complete the TODO in the previous cell, run it, and then run this cell again."

model = load_model(MODEL_PATH)
oneshot(model=model, recipe=recipe_w8a16)

save_dir = OUTPUT_DIR / "granite-4.0-350m-W8A16"
model.save_pretrained(save_dir, save_compressed=True)
tokenizer.save_pretrained(save_dir)
del model

results["W8A16"] = reload_and_test(save_dir, tokenizer, PROMPTS)

### 6. Recipe 2: W4A16

Write the second recipe yourself.
Use the same `targets` and `ignore` values as the W8A16 recipe, with the scheme for 4-bit integer weights and 16-bit activations.

In [ ]:
recipe_w4a16 = None

# TODO: replace the previous line with a QuantizationModifier recipe that quantizes
# all Linear layers with the W4A16 scheme and ignores the lm_head layer.

In [ ]:
assert recipe_w4a16 is not None, "Complete the TODO in the previous cell, run it, and then run this cell again."

model = load_model(MODEL_PATH)
oneshot(model=model, recipe=recipe_w4a16)

save_dir = OUTPUT_DIR / "granite-4.0-350m-W4A16"
model.save_pretrained(save_dir, save_compressed=True)
tokenizer.save_pretrained(save_dir)
del model

results["W4A16"] = reload_and_test(save_dir, tokenizer, PROMPTS)

### 7. Estimate GPU memory

Before you request GPU resources, estimate the VRAM that each model needs on a GPU-backed serving runtime, such as vLLM, as the sum of two parts:

* **Weights**: the packed weights occupy about the same space in GPU memory as on disk.
* **KV cache**: the attention keys and values of every token that the model is processing.
The KV cache stays at 16-bit precision for all three models, because W8A16 and W4A16 quantize only the weights.

KV cache bytes = 2 (keys and values) × attention layers × KV heads × head dimension × 2 bytes × tokens

The estimate assumes the workload in the following cell, and is the minimum VRAM that the model needs.
Real deployments also need memory for activations and for serving runtime overhead, such as the CUDA context and CUDA graphs.
Serving runtimes such as vLLM also reserve most of the remaining GPU memory for the KV cache, so GPU monitoring tools show higher memory use than this estimate.

In [ ]:
CONCURRENT_REQUESTS = 8
TOKENS_PER_REQUEST = 4096

config = json.loads((MODEL_PATH / "config.json").read_text())
attention_layers = config.get("layer_types", ["attention"] * config["num_hidden_layers"]).count("attention")
head_dim = config["hidden_size"] // config["num_attention_heads"]
kv_bytes_per_token = 2 * attention_layers * config["num_key_value_heads"] * head_dim * 2
kv_cache_gb = kv_bytes_per_token * CONCURRENT_REQUESTS * TOKENS_PER_REQUEST / 1024**3

print(f"KV cache per token: {kv_bytes_per_token / 1024:.0f} KiB")
print(f"KV cache for {CONCURRENT_REQUESTS} requests x {TOKENS_PER_REQUEST} tokens: {kv_cache_gb:.2f} GB")

### 8. Compare the models

Compare the size, the estimated VRAM, and the answers of the three models.

In [ ]:
import pandas as pd

baseline_mb = results["BF16"]["size_mb"]

rows = []
for name, r in results.items():
    weights_gb = r["size_mb"] / 1024
    rows.append({
        "Model": name,
        "Weights (MB)": round(r["size_mb"]),
        "Reduction (%)": round((1 - r["size_mb"] / baseline_mb) * 100, 1),
        "Weights VRAM (GB)": round(weights_gb, 2),
        "Est. VRAM (GB)": round(weights_gb + kv_cache_gb, 2),
    })

display(pd.DataFrame(rows))

for i, prompt in enumerate(PROMPTS):
    print(f"\nQ: {prompt}")
    for name, r in results.items():
        answer = " ".join(r["answers"][i].split())
        print(f"  {name:<6} {answer[:100]}")

Your numbers might be slightly different. Notice the following results:

* The W4A16 model reduces the weights size by about half, not by three quarters.
The embedding layer, which the model shares with the `lm_head` output layer, holds about 100 million of the 350 million parameters, and stays at the original precision.
* For this small model, the KV cache requires more memory than the weights.
Quantizing the weights therefore reduces the total VRAM estimate less than it reduces the weights size.
* All three models answer the test prompts correctly.
The W8A16 answers are almost identical to the original answers, and the W4A16 answers use different wording.

### 9. Scale the estimate to a larger model

Estimate the weights VRAM of a 70 billion parameter model with the same quantization schemes.

In [ ]:
PARAMETERS = 70e9  # 70B parameters

model_column = f"Model ({PARAMETERS / 1e9:g}B)"

rows = []
for name, bits in [("BF16", 16), ("W8A16", 8), ("W4A16", 4)]:
    weights_gb = PARAMETERS * bits / 8 / 1024**3
    rows.append({model_column: name, "Weights VRAM (GB)": round(weights_gb)})

display(pd.DataFrame(rows))

The BF16 weights of a 70 billion parameter model (about 130 GB) do not fit on a single 80 GB GPU.
The W4A16 weights (about 33 GB) fit on a single 80 GB GPU, with memory left for the KV cache.
Change `PARAMETERS` to `8e9` and run the cell again to compare with an 8 billion parameter model.